## Void Striker

Destroy enemies to unlock each section of the **Backgrounds** lesson.
Every visual trick in this game — the drifting nebula, the twinkling stars, the glowing dust — is a technique you will learn by playing through it.

**Shoot 5 enemies** to unlock the next lesson block. Keep going until all 4 sections are revealed.

### Controls

| Key | Action |
|---|---|
| Arrow keys / WASD | Move your ship |
| Space | Fire (triple-shot spread) |
| Background button (top center) | Cycle between Nebula, Deep Space, and Supernova scenes |


In [ ]:
%%js

// GAME_RUNNER: Void Striker | hide_edit: true, width: 100%, height: 500px

import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelVoidStriker from '/assets/js/projects/backgrounds/levels/GameLevelVoidStriker.js';
export const gameLevelClasses = [GameLevelVoidStriker];
export { GameControl };


<div id='lesson-progress-bar' style='
  display: flex;
  flex-direction: row;
  gap: 8px;
  align-items: center;
  font-family: monospace;
  background: #0a001a;
  border: 1px solid #4400aa;
  padding: 10px 14px;
  border-radius: 6px;
  margin-bottom: 10px;
  flex-wrap: wrap;
'>
  <span style='color:#aaaaff; font-size:13px; margin-right:6px;'>Progress:</span>
  <div id='sq-1' style='width:80px; height:36px; border:2px solid #4400aa; border-radius:5px; display:flex; align-items:center; justify-content:center; font-size:11px; color:#553388; background:#0d0020; text-align:center; padding:2px;'>🔒<br>Concept 1</div>
  <div id='sq-2' style='width:80px; height:36px; border:2px solid #4400aa; border-radius:5px; display:flex; align-items:center; justify-content:center; font-size:11px; color:#553388; background:#0d0020; text-align:center; padding:2px;'>🔒<br>Hack 1</div>
  <div id='sq-3' style='width:80px; height:36px; border:2px solid #4400aa; border-radius:5px; display:flex; align-items:center; justify-content:center; font-size:11px; color:#553388; background:#0d0020; text-align:center; padding:2px;'>🔒<br>Concept 2</div>
  <div id='sq-4' style='width:90px; height:36px; border:2px solid #4400aa; border-radius:5px; display:flex; align-items:center; justify-content:center; font-size:11px; color:#553388; background:#0d0020; text-align:center; padding:2px;'>🔒<br>New Features</div>
  <span id='kill-counter' style='margin-left:auto; color:#aaaaff; font-size:12px; opacity:0.7;'>0 kills</span>
</div>
<div id='lesson-progress' style='font-family: monospace; background: #0a001a; border: 1px solid #4400aa; padding: 10px 18px; border-radius: 6px; color: #aaaaff; font-size: 13px; margin-bottom: 8px;'>
  🔒 <strong>Lesson locked.</strong> Destroy 5 enemies in-game to reveal the first concept.
  <br><span style='opacity:0.6; font-size:12px;'>Keep shooting — each 5 kills unlocks the next section.</span>
</div>

<div class='lesson-block' id='lesson-4' style='display:none'>
<hr />
<h2>🔒 New Features — Implementing Two Peer Lessons in Void Striker <em>(unlock: 20 kills)</em></h2>
<p style='font-size:14px; opacity:0.85; margin-top:-8px;'><em>Bringing gravity and a chasing boss enemy into a top-down space shooter — adapting platformer mechanics for an arcade context.</em></p>

<h3>The Goal</h3>
<p>Two ideas from peer lessons — the <strong>Gravity System</strong> (per-frame <code>verticalVelocity</code> updates) and the <strong>Chasing Enemy</strong> from the <em>ocean</em> lesson (a per-frame <code>update()</code> that uses <code>Math.atan2</code> to home in on the player) — were both designed for side-scrolling platformers. Void Striker has no platformer primitives, so the challenge is keeping the lesson math intact while bolting it onto a free-flying ship and a top-down arcade flow.</p>

<h3>Feature 1 — Gravity-Driven Asteroids</h3>
<p>The peer lesson's gravity model is a single, very clean idea: maintain <code>verticalVelocity</code>, subtract <code>gravityAcceleration</code> from it every frame, then convert to engine velocity by flipping the sign. Originally it's used on the player to produce platformer-style jump arcs. In Void Striker, asteroids previously had a constant fall speed — replacing that with the gravity loop makes them <em>accelerate</em> downward, just like real falling rocks.</p>

<p><strong>Spawn-time properties (matching the lesson's <code>playerData</code> shape):</strong></p>
<pre><code class="language-javascript">asteroids.push({
  x: rand(40, W - 40),
  y: -40 - i * 60,
  r: rand(18, 34 + Math.min(wave * 1.5, 20)),
  // Lesson properties: positive verticalVelocity = up
  verticalVelocity:    rand(0.2, 0.8),
  gravityAcceleration: 0.05 + wave * 0.005,
  terminalVelocity:    3.5 + wave * 0.15,
  vx: rand(-0.6, 0.6),
  rot: 0,
  rotV: rand(-0.03, 0.03),
  /* … shape points … */
});</code></pre>

<p><strong>Per-frame gravity loop</strong> — straight from the lesson, with one addition (terminal velocity) so asteroids don't fall faster than the player can react:</p>
<pre><code class="language-javascript">asteroids.forEach(a =&gt; {
  // 1. Subtract gravity from vertical velocity (pulls asteroid down)
  a.verticalVelocity -= a.gravityAcceleration;
  // 2. Clamp at terminal velocity
  if (-a.verticalVelocity &gt; a.terminalVelocity) {
    a.verticalVelocity = -a.terminalVelocity;
  }
  // 3. Convert to engine velocity (flip sign): negative = downward
  a.y   += -a.verticalVelocity * worldSpeed;
  a.x   += a.vx                * worldSpeed;
  a.rot += a.rotV               * worldSpeed;
  if (a.x &lt; 20 || a.x &gt; W - 20) a.vx *= -1;
});</code></pre>

<h3>Feature 2 — The Boss Alien (Chasing Enemy)</h3>
<p>The <em>ocean</em> peer lesson teaches a chasing enemy with a single repeating <code>update()</code>: each frame, find the player, compute the angle with <code>Math.atan2</code>, and step toward them along <code>(cos(angle), sin(angle))</code>. Void Striker grafts that onto a wave-3 boss fight: a tanky <strong>Boss Alien</strong> that homes in on the ship, has a red HP bar pinned to the top of the screen, and slows the rest of the world while it's loose. Killing it counts as 5 kills toward the unlock counter.</p>

<p><strong>Step 1 — Spawn the boss on wave 3.</strong> The boss carries its own <code>hp</code>, <code>maxHp</code>, and <code>speed</code>. Spawning the boss also drops the global <code>worldSpeed</code> multiplier so enemies, asteroids, and enemy bullets all tick at 40% — the player gets a moment to focus on the new threat without trash mobs piling up:</p>
<pre><code class="language-javascript">function spawnBoss() {
  boss = {
    x: W / 2, y: -80,
    r: 40,
    hp: 30, maxHp: 30,
    speed: 1.7,
    pulse: 0,
  };
  worldSpeed = 0.4;                             // slow everything else down
  document.getElementById('vs-boss-bar').style.display = 'block';
}

function spawnWave() {
  if (wave === 3 &amp;&amp; !boss) spawnBoss();
  /* … rest of wave spawn … */
}</code></pre>

<p><strong>Step 2 — The chase update (peer lesson math).</strong> The lesson's whole insight: every frame, compute <code>dy</code> and <code>dx</code> to the player, feed them into <code>Math.atan2(dy, dx)</code> (note the order — <code>dy</code> first!), and step toward the player along <code>(cos, sin)</code>. The HP bar's CSS width gets repainted from <code>boss.hp</code>:</p>
<pre><code class="language-javascript">function updateBoss() {
  if (!boss) return;
  boss.pulse += 0.08;

  // Distance + angle to player (peer lesson: distance/atan2)
  const dx = ship.x - boss.x;
  const dy = ship.y - boss.y;
  const angle = Math.atan2(dy, dx);             // careful: dy first, then dx
  boss.x += Math.cos(angle) * boss.speed;
  boss.y += Math.sin(angle) * boss.speed;

  // Reflect HP into the on-screen bar
  const fill = document.getElementById('vs-boss-fill');
  if (fill) fill.style.width = (Math.max(0, boss.hp) / boss.maxHp * 100) + '%';
}</code></pre>

<p><strong>Step 3 — Damage and death.</strong> Bullets check the boss <em>first</em> in the collision loop so it absorbs hits even when overlapping smaller enemies. When <code>boss.hp</code> hits zero, <code>killBoss()</code> hands out 5 kills, restores <code>worldSpeed</code> to 1, and hides the bar:</p>
<pre><code class="language-javascript">// Inside the bullet collision loop:
if (boss) {
  const dx = b.x - boss.x, dy = b.y - boss.y;
  if (Math.sqrt(dx * dx + dy * dy) &lt; boss.r + 4) {
    boss.hp--;
    b.life = 0;
    if (boss.hp &lt;= 0) killBoss();
  }
}

function killBoss() {
  if (!boss) return;
  spawnExplosion(boss.x, boss.y, '#ff4477');
  totalKills += 5;                              // boss = 5 kills
  window.dispatchEvent(new CustomEvent('vs-kills', { detail: { total: totalKills } }));
  boss = null;
  worldSpeed = 1;                               // resume full speed
  document.getElementById('vs-boss-bar').style.display = 'none';
}</code></pre>

<p><strong>Step 4 — Directional shooting with the arrow keys.</strong> A boss that homes in on the ship will inevitably get <em>behind</em> it, where the old upward-only triple-shot was useless. WASD now controls movement only, and the arrow keys fire bullets in the pressed direction. Combining arrows produces diagonals — Up+Right shoots up-right — so the player can defend any side:</p>
<pre><code class="language-javascript">function updateShip() {
  // WASD = movement only
  if (keys['a'] || keys['A']) ship.x -= ship.speed;
  if (keys['d'] || keys['D']) ship.x += ship.speed;
  if (keys['w'] || keys['W']) ship.y -= ship.speed;
  if (keys['s'] || keys['S']) ship.y += ship.speed;
  /* … bounds + cooldown ticks … */

  // Arrow keys = directional shooting
  if (ship.shootCooldown === 0) {
    let sx = 0, sy = 0;
    if (keys['ArrowLeft'])  sx -= 1;
    if (keys['ArrowRight']) sx += 1;
    if (keys['ArrowUp'])    sy -= 1;
    if (keys['ArrowDown'])  sy += 1;
    if (sx !== 0 || sy !== 0) {
      fireDirected(sx, sy);
      ship.shootCooldown = 12;
    }
  }
}

function fireDirected(sx, sy) {
  const len = Math.hypot(sx, sy) || 1;
  const speed = 11;
  bullets.push({
    x: ship.x + (sx / len) * 18,                // spawn just outside the ship
    y: ship.y + (sy / len) * 18,
    vx: (sx / len) * speed,
    vy: (sy / len) * speed,
    life: 60,
  });
}</code></pre>

<h3>The <code>worldSpeed</code> Trick</h3>
<p>Every other moving object in <code>updateEnemies</code> multiplies its delta by <code>worldSpeed</code>. Spawning the boss flips it to <code>0.4</code>; killing the boss flips it back to <code>1</code>. One scalar — and the whole arena shifts gear without rewriting any other update function:</p>
<pre><code class="language-javascript">enemies.forEach(e =&gt; {
  e.y += e.speed * worldSpeed;
  e.x += e.vx    * worldSpeed;
  if (e.shootTimer !== Infinity) e.shootTimer -= worldSpeed;
});
enemyBullets.forEach(b =&gt; { b.x += b.vx * worldSpeed; b.y += b.vy * worldSpeed; });</code></pre>

<h3>Why This Adaptation Matters</h3>
<p>Both lessons were written for a platformer with discrete grounded states, jump latches, and AABB collisions. Void Striker has none of those primitives. But the <em>core math</em> in each lesson — Euler integration for gravity, <code>Math.atan2</code> for pursuit — is genre-agnostic. The chasing-enemy lesson even warns about a classic bug: writing <code>Math.atan2(dx, dy)</code> instead of <code>Math.atan2(dy, dx)</code>. That trap is identical here; the boss would crab sideways instead of homing in. The takeaway: when a peer lesson teaches a technique, the technique is rarely tied to the original game's structure. The control flow around it is scaffolding; the algorithm in the middle is the part you actually reuse.</p>

<h3>Wiring It Into the Game Loop</h3>
<pre><code class="language-javascript">// Every frame, while playing:
updateShip();
updateBullets();
updateEnemies();           // gravity loop applied to asteroids here
updateBoss();              // atan2 chase + HP bar update
updateParticles();
checkCollisions();         // bullet→boss, ship↔boss, wave-clear gated on !boss

drawEnemies();
drawBoss();                // glowing alien with tentacles
drawBullets();
drawParticles();
drawShip();</code></pre>

<p style='margin-top:24px; padding:12px 16px; background:#0d2b00; border:1px solid #00cc44; border-radius:6px; color:#44ff88; font-family:monospace; font-size:13px;'>
🎉 <strong>Lesson complete!</strong> Two peer lessons — gravity and chasing-enemy pursuit — bolted onto a game built on a totally different foundation, plus a directional-shooting tweak so the new boss can be fought from any angle.</p>

<p style='margin-top:18px; padding:14px 18px; background:#1a0d2b; border:1px solid #aa44ff; border-radius:6px; color:#dda0ff; font-family:monospace; font-size:13px; text-align:center;'>
✨ <strong>Credit to the peer lesson authors</strong> — <em>Team Sorcerers</em> for the <strong>Gravity System</strong> and the <em>ocean</em> team for the <strong>Chasing Enemy</strong> — whose work powers the features you just played through. ✨
</p>
</div>

<script>
(function() {
  const milestones = [5, 10, 15, 20];
  let unlocked = 0;

  function updateSquare(n, label) {
    var sq = document.getElementById('sq-' + n);
    if (sq) {
      sq.style.background = '#0d2b00';
      sq.style.borderColor = '#00cc44';
      sq.style.color = '#44ff88';
      sq.innerHTML = '&#x2705;<br>' + label;
    }
  }

  const labels = ['Concept 1', 'Hack 1', 'Concept 2', 'New Features'];

  function unlock(n) {
    var block = document.getElementById('lesson-' + n);
    if (block) {
      block.style.display = 'block';
      block.style.animation = 'fadeIn 0.6s ease';
      if (typeof hljs !== 'undefined') {
        block.querySelectorAll('pre code').forEach(function(el) {
          hljs.highlightElement(el);
        });
      }
    }
    var h2s = document.querySelectorAll('h2');
    h2s.forEach(function(h) {
      const label = labels[n-1];
      if (h.textContent.includes(label)) {
        h.textContent = h.textContent.replace('🔒', '✅');
      }
    });
    updateSquare(n, labels[n-1]);
    var progress = document.getElementById('lesson-progress');
    if (progress) {
      if (n < 4) {
        progress.innerHTML = '✅ ' + labels[n-1] + ' unlocked! Destroy ' + milestones[n] + ' total enemies for the next one.';
      } else {
        progress.innerHTML = '🎉 <strong>All sections unlocked!</strong> You finished the Backgrounds lesson.';
      }
    }
  }

  window.addEventListener('vs-kills', function(e) {
    var total = e.detail && e.detail.total;
    if (!total) return;
    var counter = document.getElementById('kill-counter');
    if (counter) counter.textContent = total + ' kill' + (total === 1 ? '' : 's');
    milestones.forEach(function(m, idx) {
      if (total >= m && unlocked <= idx) {
        unlocked = idx + 1;
        unlock(idx + 1);
      }
    });
  });

  var style = document.createElement('style');
  style.textContent = '@keyframes fadeIn { from { opacity: 0; transform: translateY(10px); } to { opacity: 1; transform: translateY(0); } }';
  document.head.appendChild(style);
})();
</script>